In [ ]:
#Analyse the dataset to answer which bidder has the highest win rate (wins / total_bid)*100. 

In [1]:
import pandas as pd
import numpy as np
df=pd.read_csv("analysis_id(in).csv")

In [15]:
df_bidder=df.groupby('bidder')[['response_count','prebid_win_count']].sum().reset_index()
df_bidder=df_bidder.iloc[1:].reset_index(drop=True)
df_bidder['win_rate']=np.where(df_bidder['prebid_win_count']>0,df_bidder['prebid_win_count']*100/df_bidder['response_count'],0)
df_bidder.sort_values(by='win_rate',ascending=False)

,bidder,response_count,prebid_win_count,win_rate
16,undertone,104,52.0,50.000000
14,teads,700,323.0,46.142857
9,pulsepoint,191,71.0,37.172775
12,sharethrough,332,104.0,31.325301
6,onemobile,2726,759.0,27.842993
10,rhythmone,655,160.0,24.427481
1,appnexus,830,175.0,21.084337
2,criteo,3181,587.0,18.453317
0,33across,886,136.0,15.349887
3,emx_digital,393,56.0,14.249364


In [16]:
df_bidder.sort_values(by='win_rate',ascending=False).head(2)

,bidder,response_count,prebid_win_count,win_rate
16,undertone,104,52.0,50.000000
14,teads,700,323.0,46.142857


In [17]:
df.columns

Index(['Unnamed: 0', 'date_hour', 'client', 'device_type', 'time_zone',
       'os_name', 'browser_name', 'ad_unit', 'size', 'bidder', 'bid_range',
       'media_type', 'request_count', 'response_count', 'prebid_win_count',
       'win_count', 'sum_bid', 'sum_time_to_respond', 'median', 'min_bid',
       'max_bid', 'avg_bid', 'sum_2nd_highest_bid', 'sum_prebid_winning_bid',
       'sum_winning_bid', 'sum_nobid', 'sum_timeout'],
      dtype='object')

In [19]:
df.isna().sum()

Unnamed: 0                   0
date_hour                    0
client                       0
device_type                  0
time_zone                    0
os_name                      0
browser_name                 0
ad_unit                      0
size                         0
bidder                       0
bid_range                    0
media_type                   0
request_count                0
response_count               0
prebid_win_count          3833
win_count                 4814
sum_bid                   1210
sum_time_to_respond       1210
median                    3240
min_bid                   1210
max_bid                   1210
avg_bid                   1210
sum_2nd_highest_bid       4266
sum_prebid_winning_bid    3833
sum_winning_bid           4814
sum_nobid                 5000
sum_timeout               5000
dtype: int64

Part 1: Describe how you would build a logistic regression model to predict the probability of a bid winning (win) based on the other columns. What features would you select and why?
Part 2: Based on the following results from a logistic regression model  trained on the dataset, interpret the coefficients:
bid_amount: Coefficient = 0.8
time_to_bid: Coefficient = -0.4
ad_unit_A: Coefficient = 0.3 (reference category: ad_unit_B)

PART - A
So,first i will divide the dataset into train and test parts using(train_test_split), we will not shuffle this as the dataset is time series type,so we may introduce bias into  it.then would deal with all the null values from the training dataset,df_train.fillna({'col1':df['col1'].median()}) and testing dataset separately ,to prevent data leakage. Then,we will select the features we want,and encode the categorical features using One hott encoding : pd.get_dummies(drop_first=True) to remove multi-linearity part ,or frequency encoding if types of categorical data is much more(as 100 types means 100*100 for one hot encoding).and do standardization for numerical data to scale the numerical data ,so that sum_time_to_respond =6614 ,and prebid_win_count=1 ,are on same scale,and one feature does not overpower the other.

Features list:
date_hour:time at which bid takes place is important ,to know the time where bid is most likely to be clicked
device_type : normally mobiles have high probability of win rate,so,tablet,mobile,desktop,type matters                 
time_zone : people are ususally more active in evenings,late nights,so,time zone tells what time is in their area          
ad_unit: this determines the place of ad placement ,so,eg: top right can be mistakenly be clicked in mobile
size     : this is important ,as this determines the size of ad,large ad has large area to be clicked                            
bidder       : this tells which company is the bidder,and we can track its history to know their characterisitics
bid_range      :  we can tell whether it will succeed or not based on past data analysis
request_count   : how many ads do they want to be placed
response_count  : how many ads they are bidding for
sum_time_to_respond : whether they are able to beat the latency or not
prebid_win_count  : how many bids they won,we can calculate the win rate ,and know their past data    
min_bid        : accoridng to past will they succeed
max_bid       :we can know their budget ,and whether it iss sufficient or not      
sum_prebid_winning_bid  : this can help in knowing the bid winning amount  

Part 2: Based on the following results from a logistic regression model  trained on the dataset, interpret the coefficients:
bid_amount: Coefficient = 0.8
time_to_bid: Coefficient = -0.4
ad_unit_A: Coefficient = 0.3 (reference category: ad_unit_B)

PART-2:


bid_amount: Coefficient = 0.8 means that it can increase the log-odds by 0.8 times,or P(click)/P(no click) will be e^0.8 times increased (P is probability)

time_to_bid: Coefficient = -0.4 means that it can decrease the log-odds by 0.4 times,or P(click)/P(no click) will be e^-0.4 times increased

ad_unit_A: Coefficient = 0.3 (reference category: ad_unit_B) means placing the ad at ad_unit_A  can increase the log-odds of a ad to be clicked by 0.3 times that of ad being placed at ad_unit_B i.e ,  P(click)/P(no click) will be e^0.3 times increased when we place ad from B to A unit
Note : z= w1x1+w2x2+c
z=log(p/1-p)
proabibility of a ad clicked=sigmoid(1/1+e^-z)
